# Facial Recognition Model
This section covers the entire workflow for creating the facial recognition model. We will load the features extracted from the images, train a classifier to identify individuals, and thoroughly evaluate its performance.

## Step 1: Library Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report, confusion_matrix

## Step 2: Load and Prepare the Data
Goal: Load the `image_features.csv` dataset. We will then separate it into features (`X`) and labels (`y`), encode the text-based labels into numbers, scale the features for optimal model performance, and finally split the data into training and testing sets.

In [4]:
def load_and_prepare_data(features_path):
    """Loads, preprocesses, and splits the image feature data."""
    print("--- Loading and Preparing Data ---")
    df = pd.read_csv(features_path)
    
    X = df.drop(columns=['person', 'augmentation_type'])
    y = df['person']

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("✅ Data prepared successfully.")
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, label_encoder

def train_model(X_train, y_train):
    """Initializes and trains a RandomForestClassifier."""
    print("\n--- Training RandomForest Model ---")
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    print("✅ Model training complete.")
    return model

def evaluate_model(model, X_test, y_test, encoder):
    """Evaluates the model and prints performance metrics."""
    print("\n--- Evaluating Model Performance ---")
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    loss = log_loss(y_test, y_pred_proba)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1:.4f}")
    print(f"Log Loss: {loss:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.title('Confusion Matrix - Facial Recognition')
    plt.xlabel('Predicted Person')
    plt.ylabel('Actual Person')
    plt.show()

def save_artifacts(model, scaler, encoder, output_dir):
    """Saves the model, scaler, and encoder to disk."""
    print("\n--- Saving Model Artifacts ---")
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(model, os.path.join(output_dir, 'face_recognition_model.pkl'))
    joblib.dump(scaler, os.path.join(output_dir, 'face_recognition_scaler.pkl'))
    joblib.dump(label_encoder, os.path.join(output_dir, 'face_recognition_encoder.pkl'))
    print(f"✅ Artifacts saved to '{output_dir}'")